# Week 03: pandas cơ bản và bảng mô tả

Mục tiêu hôm nay không phải là học hết pandas. Mục tiêu là tạo **một bảng mô tả có thể đưa vào paper**: đọc CSV, chọn cột, lọc dòng, tính `gain_score`, dùng `groupby`, rồi viết caption.

## 1. Cài đặt ý tưởng

`pandas` là thư viện giúp Python làm việc với bảng dữ liệu. Một `DataFrame` giống như bảng CSV đang nằm trong bộ nhớ để Python có thể chọn cột, lọc dòng và tóm tắt theo nhóm.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd

THIS_WEEK = "week-03-pandas-descriptive-tables"


def find_week_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "weeks" / THIS_WEEK,
        Path.cwd().parent,
        Path.cwd().parent / "weeks" / THIS_WEEK,
    ]
    for candidate in candidates:
        if candidate.name == THIS_WEEK and (candidate / "data/raw").exists():
            return candidate
        if (candidate / "data/raw/week03_tcsol_scores.csv").exists():
            return candidate
    week_dir = Path.cwd() / "weeks" / THIS_WEEK
    week_dir.mkdir(parents=True, exist_ok=True)
    return week_dir


WEEK_DIR = find_week_dir()
DATA_PATH = WEEK_DIR / "data/raw/week03_tcsol_scores.csv"
OUTPUT_DIR = WEEK_DIR / "outputs/tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    source_url = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/76964a1c847cff1906015d1a31875bdf30ccb910/weeks/week-03-pandas-descriptive-tables/data/raw/week03_tcsol_scores.csv"
    urlretrieve(source_url, DATA_PATH)

print("pandas version:", pd.__version__)
print("Week folder:", WEEK_DIR)
print("Data file:", DATA_PATH)


pandas version: 2.2.3
Week folder: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-03-pandas-descriptive-tables
Data file: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-03-pandas-descriptive-tables/data/raw/week03_tcsol_scores.csv


## 2. Đọc CSV thành DataFrame

Sau khi đọc file, luôn kiểm tra: có bao nhiêu dòng, bao nhiêu cột, và các tên cột là gì?

In [2]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print()
print(df.head().to_string(index=False))

Shape: (32, 7)
Columns: ['learner_id', 'class_group', 'activity_focus', 'pre_score', 'post_score', 'attendance_hours', 'completed']

learner_id class_group activity_focus  pre_score  post_score  attendance_hours completed
      S001           A  measure_words         58          72               5.5       yes
      S002           A  measure_words         62          75               6.0       yes
      S003           A  measure_words         65          78               6.0       yes
      S004           A  measure_words         70          80               5.0       yes
      S005           A  measure_words         55          68               4.5       yes


## 3. Chọn cột cần cho câu hỏi

Câu hỏi nhỏ: nhóm hoạt động nào có mức tăng điểm mô tả cao nhất? Vì vậy ta chỉ giữ các cột cần cho câu hỏi và caption.

In [3]:
score_columns = [
    "learner_id",
    "class_group",
    "activity_focus",
    "pre_score",
    "post_score",
    "completed",
]

scores = df[score_columns]
print(scores.head(8).to_string(index=False))

learner_id class_group activity_focus  pre_score  post_score completed
      S001           A  measure_words         58          72       yes
      S002           A  measure_words         62          75       yes
      S003           A  measure_words         65          78       yes
      S004           A  measure_words         70          80       yes
      S005           A  measure_words         55          68       yes
      S006           A  measure_words         60          73       yes
      S007           A  measure_words         66          76       yes
      S008           A  measure_words         59          70       yes


## 4. Lọc dòng hoàn thành

Lọc dòng không phải là sửa file gốc. Ta tạo một bảng mới tên `complete` để summary rõ hơn.

In [4]:
complete = scores[scores["completed"] == "yes"].copy()

print("Rows in raw data:", len(scores))
print("Completed rows:", len(complete))
print("Excluded rows:", len(scores) - len(complete))

Rows in raw data: 32
Completed rows: 31
Excluded rows: 1


## 5. Tạo cột `gain_score`

`gain_score` là điểm sau học trừ điểm trước học. Đây là cột tính thêm, không phải cột có sẵn trong CSV raw.

In [5]:
complete = complete.assign(
    gain_score=complete["post_score"] - complete["pre_score"]
)

print(complete[["learner_id", "activity_focus", "pre_score", "post_score", "gain_score"]].head(8).to_string(index=False))

learner_id activity_focus  pre_score  post_score  gain_score
      S001  measure_words         58          72          14
      S002  measure_words         62          75          13
      S003  measure_words         65          78          13
      S004  measure_words         70          80          10
      S005  measure_words         55          68          13
      S006  measure_words         60          73          13
      S007  measure_words         66          76          10
      S008  measure_words         59          70          11


## 6. `groupby`: nhóm rồi tính

Đọc dòng dưới bằng lời: nhóm các dòng theo `activity_focus`, rồi tính N, điểm trung bình trước học, điểm trung bình sau học, và điểm tăng trung bình.

In [6]:
summary = (
    complete
    .groupby("activity_focus")
    .agg(
        n_learners=("learner_id", "count"),
        mean_pre=("pre_score", "mean"),
        mean_post=("post_score", "mean"),
        mean_gain=("gain_score", "mean"),
    )
    .reset_index()
)

summary[["mean_pre", "mean_post", "mean_gain"]] = summary[["mean_pre", "mean_post", "mean_gain"]].round(1)
summary = summary.sort_values("mean_gain", ascending=False)

print(summary.to_string(index=False))

    activity_focus  n_learners  mean_pre  mean_post  mean_gain
     measure_words           8      61.9       74.0       12.1
result_complements           8      58.9       70.9       12.0
        word_order           7      63.4       71.9        8.4
 vocabulary_review           8      72.5       80.4        7.9


## 7. Xuất bảng mô tả

Bảng này là artifact của Week 03. Nó có thể được mở trong Excel, đưa vào Word, hoặc dùng để viết caption.

In [7]:
output_path = OUTPUT_DIR / "week03_group_summary.csv"
summary.to_csv(output_path, index=False)
print("Saved summary table to:", output_path)

Saved summary table to: /Users/mitu/Desktop/work/projects/tcsol-python-research-syllabus/weeks/week-03-pandas-descriptive-tables/outputs/tables/week03_group_summary.csv


## 8. Caption cho paper

Caption tốt không chỉ nói bảng có gì. Nó nói nguồn, N, đơn vị đo, cách tính, pattern chính và giới hạn.

In [8]:
top_row = summary.iloc[0]
second_row = summary.iloc[1]
caption = (
    f"Table 1. The {top_row['activity_focus']} group shows the largest descriptive "
    f"average gain ({top_row['mean_gain']} points), closely followed by "
    f"{second_row['activity_focus']} ({second_row['mean_gain']} points). "
    "Data are from an instructor-created Week 03 teaching dataset, "
    f"N = {len(complete)} completed learner records; scores are on a 0-100 scale, "
    "and gain is post_score minus pre_score. The table is descriptive: activity focus "
    "is linked to class section in this toy dataset, so the result does not establish a causal effect."
)

source_note = "Source note: instructor-created synthetic Week 03 dataset, accessed 2026-06-03."

print(caption)
print(source_note)


Table 1. The measure_words group shows the largest descriptive average gain (12.1 points), closely followed by result_complements (12.0 points). Data are from an instructor-created Week 03 teaching dataset, N = 31 completed learner records; scores are on a 0-100 scale, and gain is post_score minus pre_score. The table is descriptive: activity focus is linked to class section in this toy dataset, so the result does not establish a causal effect.
Source note: instructor-created synthetic Week 03 dataset, accessed 2026-06-03.


## 9. Bài tập nhỏ trong notebook

Đổi `group_variable` thành `class_group`, chạy lại cell, rồi viết một câu so sánh. Đây là bài tập **sao chép rồi sửa**.

In [9]:
group_variable = "activity_focus"  # Try: "class_group"

practice_summary = (
    complete
    .groupby(group_variable)
    .agg(
        n_learners=("learner_id", "count"),
        mean_gain=("gain_score", "mean"),
    )
    .reset_index()
)
practice_summary["mean_gain"] = practice_summary["mean_gain"].round(1)
print(practice_summary.sort_values("mean_gain", ascending=False).to_string(index=False))

    activity_focus  n_learners  mean_gain
     measure_words           8       12.1
result_complements           8       12.0
        word_order           7        8.4
 vocabulary_review           8        7.9


## 10. Diễn giải 120-160 từ

Viết trong Markdown cell hoặc Word. Dùng scaffold này:

- Câu 1: nêu câu hỏi và metric chính (`mean_gain`).
- Câu 2: nêu pattern chính và N.
- Câu 3: so sánh nhóm đứng đầu với nhóm gần nhất.
- Câu 4: nêu limitation: activity focus đang gắn với class section, nên không kết luận nhân quả.
- Câu 5: nêu next step cho Week 04 hoặc thu thập dữ liệu.

Mẫu ngắn: Trong dataset Week 03, nhóm `measure_words` có mean gain cao nhất (12.1 điểm), gần sát `result_complements` (12.0 điểm). Bảng dựa trên N = 31 completed learner records. Vì activity focus gần như đi cùng class section, kết quả chỉ mô tả pattern trong dữ liệu này và không chứng minh hoạt động nào hiệu quả hơn. Bước tiếp theo là làm sạch dữ liệu và kiểm tra missing values trước khi trực quan hóa hoặc so sánh sâu hơn.